In [18]:
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plot_utils as pu
from plot_utils import create_figure, save_figure
from huggingface_hub import snapshot_download

In [20]:
bucket_df = []
local_dir = snapshot_download(
    repo_id="eth-easl/swissai-serving-trace",
    repo_type="dataset",
    allow_patterns=[
        "qwen3-32b-bucket-reuse.jsonl",
        "qwen380b_thinking_bucket-reuse.jsonl",
        "qwen380b_instruct_bucket-reuse.jsonl",
        "llama3-70b_bucket-reuse.jsonl",
        "apertus-70b-bucket-reuse.jsonl"
    ],
)

with open(f"{local_dir}/qwen3-32b-bucket-reuse.jsonl", "r") as f:
    data = [json.loads(line) for line in f]
qwen_df = pd.DataFrame(data)
qwen_df['model']= 'Qwen/Qwen3-32B'
apertus_df = []
with open(f"{local_dir}/apertus-70b-bucket-reuse.jsonl", "r") as f:
    data = [json.loads(line) for line in f]
apertus_df = pd.DataFrame(data)
apertus_df['model']= 'swissai/Apertus-70B'

with open(f"{local_dir}/qwen380b_thinking_bucket-reuse.jsonl", "r") as f:
    data = [json.loads(line) for line in f]
qwen380b_thinking_df = pd.DataFrame(data)
qwen380b_thinking_df['model']= 'Qwen/Qwen3-Next-80B-A3B-Thinking'

with open(f"{local_dir}/qwen380b_instruct_bucket-reuse.jsonl", "r") as f:
    data = [json.loads(line) for line in f]
qwen380b_instruct_df = pd.DataFrame(data)
qwen380b_instruct_df['model']= 'Qwen/Qwen3-Next-80B-A3B-Instruct'

with open(f"{local_dir}/llama3-70b_bucket-reuse.jsonl", "r") as f:
    data = [json.loads(line) for line in f]
llama3_70b_df = pd.DataFrame(data)
llama3_70b_df['model']= 'meta-llama/Llama-3.3-70B-Instruct'

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 4505.16it/s]


In [21]:
bucket_df = pd.concat([
    qwen_df,
    apertus_df,
    qwen380b_instruct_df,
    qwen380b_thinking_df
], ignore_index=True)
bucket_df.head()
bucket_df['model'].unique()

<ArrowStringArray>
[                  'Qwen/Qwen3-32B',              'swissai/Apertus-70B',
 'Qwen/Qwen3-Next-80B-A3B-Instruct', 'Qwen/Qwen3-Next-80B-A3B-Thinking']
Length: 4, dtype: str

In [22]:
print(f"models before filtering: {bucket_df['model'].unique()}")
linestyles = ['-', '--', '-.', ':']
markers = ['o', 'X', '|', 's', '^']
bucket_df["created_at"] = pd.to_datetime(bucket_df["created_at"],utc=True, format="%Y-%m-%dT%H:%M:%S.%fZ", errors="coerce")
# Drop rows where timestamp parsing failed to keep daily stats accurate
bucket_df = bucket_df.dropna(subset=["created_at"]).copy()
# Compute per-model daily average reuse percentage
daily_reuse = (
    bucket_df.assign(created_date=bucket_df["created_at"].dt.date)
    .groupby(["created_date", "model"])
    .agg(
        avg_reuse_pct=("reuse_percentage", "mean"),
        request_count=("reuse_percentage", "size"),
    )
    .reset_index()
    .sort_values(["created_date", "model"])
)
print(f"models: {daily_reuse['model'].unique()}")
# Convert created_date to datetime for lifecycle math
daily_reuse["created_date"] = pd.to_datetime(daily_reuse["created_date"])
# Compute days since first observation per model (life-cycle)
daily_reuse["days_since_start"] = daily_reuse.groupby("model")["created_date"].transform(lambda x: (x - x.min()).dt.days)

daily_reuse["pct_of_peak"] = daily_reuse.groupby("model")["avg_reuse_pct"].transform(lambda x: x)
daily_reuse["max_days"] = daily_reuse.groupby("model")["days_since_start"].transform("max")
daily_reuse.loc[daily_reuse["max_days"] == 0, "max_days"] = 1
daily_reuse["lifecycle_pct"] = (daily_reuse["days_since_start"] / daily_reuse["max_days"]) * 100

# daily_reuse['model'] = daily_reuse['model'].split('/').str[-1]
daily_reuse['model'] = daily_reuse['model'].transform(lambda x: x.split('/')[ -1])
models = daily_reuse['model'].unique()

fig, ax = create_figure(1, 1, figsize=(9, 3.75))
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for i, model in enumerate(models):
    
    mdf = daily_reuse[daily_reuse["model"] == model].sort_values("lifecycle_pct")
    ax.plot(mdf["lifecycle_pct"], mdf["pct_of_peak"], marker=markers[i % len(markers)], linestyle=linestyles[i % len(linestyles)], color=colors[i % len(colors)], linewidth=1.5, markersize=5, label=model)

ax.set_title("Reusable Prefixes over Model Lifecycle", fontsize=14)
ax.set_xlabel("Model Lifecycle (% of days observed)", fontsize=13)
ax.set_ylabel("Reusable Prefixes (%)", fontsize=13)
ax.grid(axis="y", linestyle=":")
ax.set_xlim(0, 100)
ax.set_ylim(0, 110)
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.12),
    ncol=2,
    fontsize=12,
    frameon=True,
)
sns.despine()
fig.tight_layout()
save_figure(fig, "figures/reuse_lifecycle_comparison", formats=['pdf'])

models before filtering: <ArrowStringArray>
[                  'Qwen/Qwen3-32B',              'swissai/Apertus-70B',
 'Qwen/Qwen3-Next-80B-A3B-Instruct', 'Qwen/Qwen3-Next-80B-A3B-Thinking']
Length: 4, dtype: str
models: <ArrowStringArray>
[                  'Qwen/Qwen3-32B',              'swissai/Apertus-70B',
 'Qwen/Qwen3-Next-80B-A3B-Thinking', 'Qwen/Qwen3-Next-80B-A3B-Instruct']
Length: 4, dtype: str


/tmp/ipykernel_3953774/3066604807.py:57: UserWarning: The figure layout has changed to tight
  fig.tight_layout()
<frozen importlib._bootstrap>:491: RuntimeWarning: The global interpreter lock (GIL) has been enabled to load module 'fontTools.misc.bezierTools', which has not declared that it can run safely without the GIL. To override this behavior and keep the GIL disabled (at your own risk), run with PYTHON_GIL=0 or -Xgil=0.


Saved figure: figures/reuse_lifecycle_comparison.pdf


<Figure size 2700x1125 with 1 Axes>